In [ ]:
!pip install seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
from scipy.stats import entropy

In [ ]:
train_val_df = pd.concat([
    pd.read_csv("/data/split/train.csv"),
    pd.read_csv("/data/split/val.csv")
], ignore_index=True)
consensus_df = pd.read_csv("/data/split/consensus_test.csv")
csv_path = "/data/consensus_label_distribution.csv"
df_votes = pd.read_csv(csv_path)
print(f"Loaded {len(df_votes)} rows from {csv_path} (Consensus votes)")
df_votes.head()

In [ ]:
def plot_label_distribution(df, title):
    plt.figure(figsize=(6,4))
    sns.countplot(x='label', data=df, order=df['label'].value_counts().index)
    plt.title(title)
    plt.ylabel("Count")
    plt.xlabel("Label")
    plt.show()

plot_label_distribution(train_val_df, "Train + Val Label Distribution")
plot_label_distribution(consensus_df, "Consensus Test Label Distribution")

In [ ]:
def plot_label_ratios_pie(df, name):
    ratios = df['label'].value_counts(normalize=True)

    plt.figure(figsize=(5, 5))
    plt.pie(
        ratios.values,
        labels=ratios.index,
        autopct="%.1f%%",
        startangle=90,
        wedgeprops={"edgecolor": "white"}
    )
    plt.title(f"{name} – label distribution")
    plt.axis("equal")
    plt.show()


plot_label_ratios_pie(train_val_df, "Train + Val")
plot_label_ratios_pie(consensus_df, "Consensus Test")

In [ ]:
def compare_distributions(train_val_df, consensus_df):
    tv = train_val_df['label'].value_counts(normalize=True).reset_index()
    tv.columns = ['Label', 'Train+Val']

    ct = consensus_df['label'].value_counts(normalize=True).reset_index()
    ct.columns = ['Label', 'Consensus Test']

    comp = pd.merge(tv, ct, on='Label', how='outer').fillna(0)

    comp_melted = comp.melt(
        id_vars="Label",
        var_name="Split",
        value_name="Ratio"
    )

    plt.figure(figsize=(6, 4))
    sns.barplot(data=comp_melted, x="Label", y="Ratio", hue="Split")
    plt.title("Label distribution comparison")
    plt.ylim(0, 1)
    plt.show()

compare_distributions(train_val_df, consensus_df)

In [ ]:
def imbalance_stats(df, name):
    counts = df['label'].value_counts()
    ratio = counts.max() / counts.min()

    print(f"=== {name} imbalance ===")
    print(counts)
    print(f"Imbalance ratio (max/min): {ratio:.2f}")
    print()

imbalance_stats(train_val_df, "Train + Val")
imbalance_stats(consensus_df, "Consensus Test")

In [ ]:
def kl_divergence(train_val_df, consensus_df):
    tv = train_val_df['label'].value_counts(normalize=True)
    ct = consensus_df['label'].value_counts(normalize=True)

    all_labels = sorted(set(tv.index) | set(ct.index))
    p = [tv.get(l, 0) + 1e-9 for l in all_labels]
    q = [ct.get(l, 0) + 1e-9 for l in all_labels]

    print("KL divergence (Train+Val || Consensus):", entropy(p, q))

kl_divergence(train_val_df, consensus_df)

In [ ]:
all_labels = ["Neutralis", "Pronacio", "Szupinacio"]

label_sums = df_votes[all_labels].sum()
plt.figure(figsize=(6,6))
plt.pie(label_sums, labels=all_labels, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("Set2"))
plt.title("Overall consensus label distribution")
plt.show()

majority_counts = df_votes['majority_label'].value_counts()
plt.figure(figsize=(6,6))
plt.pie(majority_counts, labels=majority_counts.index, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("Set3"))
plt.title("Majority label distribution per image")
plt.show()

df_votes['confidence'] = df_votes['majority_count'] / df_votes['total_votes']
plt.figure(figsize=(8,4))
sns.histplot(df_votes['confidence'], bins=10, kde=True, color='skyblue')
plt.title("Distribution of consensus confidence")
plt.xlabel("Confidence (majority votes / total votes)")
plt.ylabel("Number of images")
plt.show()

high_confidence = df_votes[df_votes['confidence'] >= 0.8]
print(f"Number of high-confidence images (>=0.8): {len(high_confidence)}")
high_confidence.head()

high_label_counts = high_confidence['majority_label'].value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=high_label_counts.index, y=high_label_counts.values, palette="Set2")
plt.title("Majority label distribution (high-confidence images)")
plt.ylabel("Number of images")
plt.show()



In [ ]:
"""
labels = ["Neutralis", "Pronacio", "Szupinacio"]

batch_size = 10
num_batches = (len(df_votes) + batch_size - 1)

for b in range(num_batches):
    batch_df = df_votes.iloc[b*batch_size : (b+1)*batch_size]
    
    x = range(len(batch_df))
    width = 0.2
    
    plt.figure(figsize=(max(10, len(batch_df)*1.2), 6))
    
    for i, label in enumerate(labels):
        plt.bar(
            [xi + i*width for xi in x],
            batch_df[label],
            width=width,
            label=label
        )
    
    plt.xticks([xi + width for xi in x], batch_df["image"], rotation=45, ha="right")
    plt.ylabel("Number of votes")
    plt.title(f"Consensus label distribution (images {b*batch_size+1}-{min((b+1)*batch_size, len(df_votes))})")
    plt.legend()
    plt.tight_layout()
    plt.show()
"""